# Análisis de Datos: Red de Tiendas RetailNow

## Objetivo
Analizar los datos de ventas, inventarios y satisfacción del cliente para una cadena de tiendas minoristas utilizando Pandas y Numpy.

## Descripción del Proyecto
La empresa ficticia RetailNow desea optimizar el rendimiento de sus sucursales en varias ciudades. Este notebook procesa y analiza datos de:
- **Ventas**: Información de productos vendidos, cantidades y precios
- **Inventarios**: Stock disponible de productos en cada tienda
- **Satisfacción del cliente**: Índices de satisfacción por tienda

## 1. Importar las librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente:")
print(f"Pandas versión: {pd.__version__}")
print(f"Numpy versión: {np.__version__}")

## 2. Cargar los datos desde los archivos CSV

In [ ]:
# Cargar los archivos CSV
# Nota: Ajusta las rutas según la ubicación de tus archivos

try:
    # Intentar cargar desde rutas relativas primero
    sales_df = pd.read_csv('sales.csv')
    inventories_df = pd.read_csv('inventories.csv')
    satisfaction_df = pd.read_csv('satisfaction.csv')
    print("✓ Archivos CSV cargados correctamente desde rutas relativas")
except FileNotFoundError:
    # Si no encuentra, intentar desde rutas absolutas
    try:
        sales_df = pd.read_csv('/workspace/sales.csv')
        inventories_df = pd.read_csv('/workspace/inventories.csv')
        satisfaction_df = pd.read_csv('/workspace/satisfaction.csv')
        print("✓ Archivos CSV cargados correctamente desde rutas absolutas")
    except FileNotFoundError:
        print("✗ Error: No se encontraron los archivos CSV. Verifica las rutas.")

print("\nDatos cargados exitosamente.")

## 3. Exploración inicial de los datos

In [ ]:
# Mostrar las primeras filas de cada DataFrame
print("=" * 80)
print("DATOS DE VENTAS (sales.csv)")
print("=" * 80)
print(sales_df.head())
print(f"\nForma del DataFrame: {sales_df.shape}")
print(f"Columnas: {sales_df.columns.tolist()}")
print(f"Tipos de datos:\n{sales_df.dtypes}")

In [ ]:
print("\n" + "=" * 80)
print("DATOS DE INVENTARIOS (inventories.csv)")
print("=" * 80)
print(inventories_df.head())
print(f"\nForma del DataFrame: {inventories_df.shape}")
print(f"Columnas: {inventories_df.columns.tolist()}")
print(f"Tipos de datos:\n{inventories_df.dtypes}")

In [ ]:
print("\n" + "=" * 80)
print("DATOS DE SATISFACCIÓN (satisfaction.csv)")
print("=" * 80)
print(satisfaction_df.head())
print(f"\nForma del DataFrame: {satisfaction_df.shape}")
print(f"Columnas: {satisfaction_df.columns.tolist()}")
print(f"Tipos de datos:\n{satisfaction_df.dtypes}")

## 4. Limpieza de datos

In [ ]:
# Verificar valores nulos antes de limpiar
print("Valores nulos antes de limpiar:")
print(f"\nVentas: {sales_df.isnull().sum().sum()} valores nulos")
print(f"Inventarios: {inventories_df.isnull().sum().sum()} valores nulos")
print(f"Satisfacción: {satisfaction_df.isnull().sum().sum()} valores nulos")

# Eliminar filas con valores nulos
sales_df = sales_df.dropna()
inventories_df = inventories_df.dropna()
satisfaction_df = satisfaction_df.dropna()

print("\nValores nulos después de limpiar:")
print(f"Ventas: {sales_df.isnull().sum().sum()} valores nulos")
print(f"Inventarios: {inventories_df.isnull().sum().sum()} valores nulos")
print(f"Satisfacción: {satisfaction_df.isnull().sum().sum()} valores nulos")

print("\n✓ Datos limpios y listos para análisis")

## 5. Análisis de Ventas

In [ ]:
# Crear columna de ingresos totales (Cantidad_Vendida * Precio_Unitario)
sales_df['Ingresos_Totales'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']

print("=" * 80)
print("ANÁLISIS DE VENTAS")
print("=" * 80)

# Ventas totales por tienda
print("\n1. VENTAS TOTALES POR TIENDA")
ventas_por_tienda = sales_df.groupby('ID_Tienda').agg({
    'Cantidad_Vendida': 'sum',
    'Ingresos_Totales': 'sum'
}).rename(columns={'Cantidad_Vendida': 'Total_Cantidad', 'Ingresos_Totales': 'Total_Ingresos'})

print(ventas_por_tienda)
print(f"\nTotal de ingresos en toda la red: ${ventas_por_tienda['Total_Ingresos'].sum():,.2f}")

In [ ]:
# Ventas totales por producto
print("\n2. VENTAS TOTALES POR PRODUCTO")
ventas_por_producto = sales_df.groupby('Producto').agg({
    'Cantidad_Vendida': 'sum',
    'Ingresos_Totales': 'sum'
}).rename(columns={'Cantidad_Vendida': 'Total_Cantidad', 'Ingresos_Totales': 'Total_Ingresos'})

print(ventas_por_producto)
print(f"\nProducto más vendido (por ingresos): {ventas_por_producto['Total_Ingresos'].idxmax()}")
print(f"Ingresos del producto más vendido: ${ventas_por_producto['Total_Ingresos'].max():,.2f}")

In [ ]:
# Resumen estadístico de las ventas
print("\n3. RESUMEN ESTADÍSTICO DE VENTAS")
print("\nEstadísticas de Ingresos Totales:")
print(sales_df['Ingresos_Totales'].describe())

print("\nEstadísticas de Cantidad Vendida:")
print(sales_df['Cantidad_Vendida'].describe())

In [ ]:
# Ventas por tienda y producto
print("\n4. VENTAS POR TIENDA Y PRODUCTO")
ventas_tienda_producto = sales_df.groupby(['ID_Tienda', 'Producto']).agg({
    'Cantidad_Vendida': 'sum',
    'Ingresos_Totales': 'sum'
}).reset_index()

print(ventas_tienda_producto.to_string())

## 6. Análisis de Inventarios

In [ ]:
print("\n" + "=" * 80)
print("ANÁLISIS DE INVENTARIOS")
print("=" * 80)

# Crear un DataFrame con ventas y inventarios fusionados
# Primero, agrupar ventas por tienda y producto
ventas_agrupadas = sales_df.groupby(['ID_Tienda', 'Producto']).agg({
    'Cantidad_Vendida': 'sum'
}).reset_index()

# Fusionar con inventarios
inventario_analisis = inventories_df.merge(
    ventas_agrupadas,
    on=['ID_Tienda', 'Producto'],
    how='left'
)

# Llenar valores NaN con 0 (productos sin ventas)
inventario_analisis['Cantidad_Vendida'] = inventario_analisis['Cantidad_Vendida'].fillna(0)

# Calcular rotación de inventarios (Cantidad_Vendida / Stock_Disponible)
inventario_analisis['Rotacion_Inventario'] = inventario_analisis['Cantidad_Vendida'] / inventario_analisis['Stock_Disponible']

# Calcular porcentaje de ventas respecto al stock
inventario_analisis['Porcentaje_Vendido'] = (inventario_analisis['Cantidad_Vendida'] / inventario_analisis['Stock_Disponible']) * 100

print("\n1. ROTACIÓN DE INVENTARIOS POR TIENDA Y PRODUCTO")
print(inventario_analisis.to_string())

In [ ]:
# Resumen de rotación por tienda
print("\n2. ROTACIÓN PROMEDIO DE INVENTARIOS POR TIENDA")
rotacion_por_tienda = inventario_analisis.groupby('ID_Tienda').agg({
    'Rotacion_Inventario': 'mean',
    'Porcentaje_Vendido': 'mean',
    'Stock_Disponible': 'sum',
    'Cantidad_Vendida': 'sum'
}).round(2)

rotacion_por_tienda.columns = ['Rotación_Promedio', 'Porcentaje_Vendido_Promedio', 'Stock_Total', 'Cantidad_Vendida_Total']
print(rotacion_por_tienda)

In [ ]:
# Identificar tiendas con niveles críticos de inventario (< 10% de ventas respecto al stock)
print("\n3. TIENDAS CON NIVELES CRÍTICOS DE INVENTARIO (< 10% de ventas respecto al stock)")
inventario_critico = inventario_analisis[inventario_analisis['Porcentaje_Vendido'] < 10]

if len(inventario_critico) > 0:
    print(f"\nSe encontraron {len(inventario_critico)} productos con niveles críticos de inventario:")
    print(inventario_critico[['ID_Tienda', 'Producto', 'Stock_Disponible', 'Cantidad_Vendida', 'Porcentaje_Vendido']].to_string())
    
    # Resumen por tienda
    tiendas_criticas = inventario_critico.groupby('ID_Tienda').size()
    print(f"\nTiendas con productos en nivel crítico:")
    print(tiendas_criticas)
else:
    print("\nNo se encontraron productos con niveles críticos de inventario.")

## 7. Análisis de Satisfacción del Cliente

In [ ]:
print("\n" + "=" * 80)
print("ANÁLISIS DE SATISFACCIÓN DEL CLIENTE")
print("=" * 80)

print("\n1. SATISFACCIÓN POR TIENDA")
print(satisfaction_df.to_string(index=False))

print(f"\nSatisfacción promedio en toda la red: {satisfaction_df['Satisfacción_Promedio'].mean():.2f}%")
print(f"Tienda con mayor satisfacción: Tienda {satisfaction_df.loc[satisfaction_df['Satisfacción_Promedio'].idxmax(), 'ID_Tienda']} ({satisfaction_df['Satisfacción_Promedio'].max()}%)")
print(f"Tienda con menor satisfacción: Tienda {satisfaction_df.loc[satisfaction_df['Satisfacción_Promedio'].idxmin(), 'ID_Tienda']} ({satisfaction_df['Satisfacción_Promedio'].min()}%)")

In [ ]:
# Identificar tiendas con baja satisfacción (< 60%)
print("\n2. TIENDAS CON BAJA SATISFACCIÓN (< 60%)")
baja_satisfaccion = satisfaction_df[satisfaction_df['Satisfacción_Promedio'] < 60]

if len(baja_satisfaccion) > 0:
    print(f"\nSe encontraron {len(baja_satisfaccion)} tiendas con baja satisfacción:")
    print(baja_satisfaccion.to_string(index=False))
    
    print("\n📋 RECOMENDACIONES PARA MEJORAR:")
    for idx, row in baja_satisfaccion.iterrows():
        tienda_id = row['ID_Tienda']
        satisfaccion = row['Satisfacción_Promedio']
        
        print(f"\n  Tienda {tienda_id} (Satisfacción: {satisfaccion}%):")
        
        # Analizar ventas de esta tienda
        ventas_tienda = ventas_por_tienda.loc[tienda_id]
        print(f"    - Ingresos totales: ${ventas_tienda['Total_Ingresos']:,.2f}")
        print(f"    - Cantidad vendida: {ventas_tienda['Total_Cantidad']:.0f} unidades")
        
        # Analizar inventario de esta tienda
        rotacion_tienda = rotacion_por_tienda.loc[tienda_id]
        print(f"    - Rotación promedio: {rotacion_tienda['Rotación_Promedio']:.2f}")
        print(f"    - Stock total: {rotacion_tienda['Stock_Total']:.0f} unidades")
        
        # Recomendaciones
        print(f"    - Acciones recomendadas:")
        if satisfaccion < 50:
            print(f"      • Realizar auditoría urgente de servicio al cliente")
            print(f"      • Capacitar al personal de la tienda")
        print(f"      • Mejorar la disponibilidad de productos (revisar inventarios)")
        print(f"      • Implementar programas de atención al cliente")
        print(f"      • Realizar encuestas para identificar problemas específicos")
else:
    print("\n✓ Todas las tiendas tienen un nivel de satisfacción >= 60%")

In [ ]:
# Relacionar satisfacción con rendimiento de ventas
print("\n3. RELACIÓN ENTRE SATISFACCIÓN Y RENDIMIENTO DE VENTAS")

# Crear DataFrame combinado
analisis_combinado = pd.DataFrame({
    'ID_Tienda': ventas_por_tienda.index,
    'Total_Ingresos': ventas_por_tienda['Total_Ingresos'].values,
    'Total_Cantidad': ventas_por_tienda['Total_Cantidad'].values
})

# Agregar satisfacción
analisis_combinado = analisis_combinado.merge(
    satisfaction_df[['ID_Tienda', 'Satisfacción_Promedio']],
    on='ID_Tienda'
)

print(analisis_combinado.to_string(index=False))

# Calcular correlación
correlacion = analisis_combinado['Total_Ingresos'].corr(analisis_combinado['Satisfacción_Promedio'])
print(f"\nCorrelación entre Ingresos y Satisfacción: {correlacion:.3f}")

if correlacion > 0.5:
    print("→ Correlación positiva fuerte: Mayor satisfacción está asociada con mayores ingresos")
elif correlacion > 0:
    print("→ Correlación positiva moderada: Existe relación entre satisfacción e ingresos")
else:
    print("→ Correlación débil o negativa: No hay relación clara entre satisfacción e ingresos")

## 8. Cálculos Estadísticos con Numpy

In [ ]:
print("\n" + "=" * 80)
print("CÁLCULOS ESTADÍSTICOS CON NUMPY")
print("=" * 80)

# Convertir la columna de ingresos totales a array de Numpy
ingresos_array = sales_df['Ingresos_Totales'].to_numpy()

print("\n1. ESTADÍSTICAS DE VENTAS TOTALES (usando Numpy)")
print(f"\nArray de ingresos (primeros 10 elementos): {ingresos_array[:10]}")

# Calcular mediana
mediana_ingresos = np.median(ingresos_array)
print(f"\nMediana de ingresos: ${mediana_ingresos:,.2f}")

# Calcular desviación estándar
desv_estandar = np.std(ingresos_array)
print(f"Desviación estándar: ${desv_estandar:,.2f}")

# Otras estadísticas
media = np.mean(ingresos_array)
minimo = np.min(ingresos_array)
maximo = np.max(ingresos_array)

print(f"\nMedia: ${media:,.2f}")
print(f"Mínimo: ${minimo:,.2f}")
print(f"Máximo: ${maximo:,.2f}")
print(f"Rango: ${maximo - minimo:,.2f}")
print(f"Varianza: ${np.var(ingresos_array):,.2f}")

In [ ]:
# Percentiles
print("\n2. PERCENTILES DE INGRESOS")
percentiles = [25, 50, 75, 90, 95]
for p in percentiles:
    valor = np.percentile(ingresos_array, p)
    print(f"Percentil {p}: ${valor:,.2f}")

## 9. Simulación de Proyecciones de Ventas Futuras con Numpy

In [ ]:
print("\n" + "=" * 80)
print("SIMULACIÓN DE PROYECCIONES DE VENTAS FUTURAS")
print("=" * 80)

# Establecer semilla para reproducibilidad
np.random.seed(42)

print("\n1. PARÁMETROS DE SIMULACIÓN")
print(f"Semilla (seed): 42")
print(f"Número de simulaciones: 1000")
print(f"Número de tiendas: 5")
print(f"Período de proyección: 12 meses")

# Parámetros de simulación
num_simulaciones = 1000
num_tiendas = 5
meses_proyeccion = 12

# Media y desviación estándar de ingresos actuales
media_ingresos = np.mean(ingresos_array)
desv_ingresos = np.std(ingresos_array)

print(f"\nMedia de ingresos actuales: ${media_ingresos:,.2f}")
print(f"Desviación estándar: ${desv_ingresos:,.2f}")

# Generar proyecciones aleatorias (distribución normal)
# Simulamos crecimiento del 5% anual
tasa_crecimiento = 0.05
proyecciones = np.random.normal(
    loc=media_ingresos * (1 + tasa_crecimiento),
    scale=desv_ingresos,
    size=(num_simulaciones, num_tiendas, meses_proyeccion)
)

print(f"\nForma del array de proyecciones: {proyecciones.shape}")
print(f"(Simulaciones: {proyecciones.shape[0]}, Tiendas: {proyecciones.shape[1]}, Meses: {proyecciones.shape[2]})")

In [ ]:
# Calcular estadísticas de las proyecciones
print("\n2. ESTADÍSTICAS DE LAS PROYECCIONES")

# Ingresos totales por simulación
ingresos_totales_simulados = np.sum(proyecciones, axis=(1, 2))

print(f"\nIngresos totales proyectados (12 meses, 5 tiendas):")
print(f"  Media: ${np.mean(ingresos_totales_simulados):,.2f}")
print(f"  Mediana: ${np.median(ingresos_totales_simulados):,.2f}")
print(f"  Desviación estándar: ${np.std(ingresos_totales_simulados):,.2f}")
print(f"  Mínimo: ${np.min(ingresos_totales_simulados):,.2f}")
print(f"  Máximo: ${np.max(ingresos_totales_simulados):,.2f}")

# Percentiles
print(f"\nPercentiles de ingresos totales proyectados:")
for p in [10, 25, 50, 75, 90]:
    valor = np.percentile(ingresos_totales_simulados, p)
    print(f"  Percentil {p}: ${valor:,.2f}")

In [ ]:
# Proyecciones por tienda
print("\n3. PROYECCIONES POR TIENDA (promedio de 1000 simulaciones)")

# Calcular ingresos promedio por tienda en todas las simulaciones
ingresos_por_tienda_proyectados = np.mean(np.sum(proyecciones, axis=2), axis=0)

for tienda_id in range(1, num_tiendas + 1):
    ingresos_proyectados = ingresos_por_tienda_proyectados[tienda_id - 1]
    print(f"\n  Tienda {tienda_id}:")
    print(f"    Ingresos proyectados (12 meses): ${ingresos_proyectados:,.2f}")
    
    # Comparar con ingresos actuales
    ingresos_actuales = ventas_por_tienda.loc[tienda_id, 'Total_Ingresos']
    diferencia = ingresos_proyectados - ingresos_actuales
    porcentaje_cambio = (diferencia / ingresos_actuales) * 100
    
    print(f"    Ingresos actuales: ${ingresos_actuales:,.2f}")
    print(f"    Diferencia proyectada: ${diferencia:,.2f} ({porcentaje_cambio:+.1f}%)")

In [ ]:
# Análisis de riesgo
print("\n4. ANÁLISIS DE RIESGO (Probabilidad de escenarios)")

# Calcular ingresos promedio actuales
ingresos_promedio_actual = np.mean(ingresos_array)

# Probabilidad de crecimiento
prob_crecimiento = np.sum(ingresos_totales_simulados > np.sum(ventas_por_tienda['Total_Ingresos'])) / len(ingresos_totales_simulados) * 100
print(f"\nProbabilidad de crecimiento respecto a ingresos actuales: {prob_crecimiento:.1f}%")

# Probabilidad de caída
prob_caida = np.sum(ingresos_totales_simulados < np.sum(ventas_por_tienda['Total_Ingresos']) * 0.9) / len(ingresos_totales_simulados) * 100
print(f"Probabilidad de caída > 10%: {prob_caida:.1f}%")

# Intervalo de confianza del 95%
ic_95_inferior = np.percentile(ingresos_totales_simulados, 2.5)
ic_95_superior = np.percentile(ingresos_totales_simulados, 97.5)
print(f"\nIntervalo de confianza del 95%:")
print(f"  Inferior: ${ic_95_inferior:,.2f}")
print(f"  Superior: ${ic_95_superior:,.2f}")

## 10. Resumen y Conclusiones

In [ ]:
print("\n" + "=" * 80)
print("RESUMEN EJECUTIVO Y CONCLUSIONES")
print("=" * 80)

print("\n📊 RENDIMIENTO GENERAL DE LA RED")
print(f"\n  Número de tiendas: {len(ventas_por_tienda)}")
print(f"  Número de productos: {len(ventas_por_producto)}")
print(f"  Total de transacciones: {len(sales_df)}")
print(f"  Ingresos totales: ${ventas_por_tienda['Total_Ingresos'].sum():,.2f}")
print(f"  Cantidad total vendida: {ventas_por_tienda['Total_Cantidad'].sum():.0f} unidades")

print("\n🏪 TIENDAS CON MEJOR RENDIMIENTO")
top_tiendas = ventas_por_tienda.nlargest(3, 'Total_Ingresos')
for idx, (tienda_id, row) in enumerate(top_tiendas.iterrows(), 1):
    satisfaccion = satisfaction_df[satisfaction_df['ID_Tienda'] == tienda_id]['Satisfacción_Promedio'].values[0]
    print(f"\n  {idx}. Tienda {tienda_id}")
    print(f"     Ingresos: ${row['Total_Ingresos']:,.2f}")
    print(f"     Satisfacción: {satisfaccion}%")

print("\n⚠️  TIENDAS QUE REQUIEREN ATENCIÓN")
bottom_tiendas = ventas_por_tienda.nsmallest(2, 'Total_Ingresos')
for idx, (tienda_id, row) in enumerate(bottom_tiendas.iterrows(), 1):
    satisfaccion = satisfaction_df[satisfaction_df['ID_Tienda'] == tienda_id]['Satisfacción_Promedio'].values[0]
    print(f"\n  {idx}. Tienda {tienda_id}")
    print(f"     Ingresos: ${row['Total_Ingresos']:,.2f}")
    print(f"     Satisfacción: {satisfaccion}%")
    if satisfaccion < 60:
        print(f"     ⚠️  Baja satisfacción - Requiere intervención urgente")

print("\n💡 RECOMENDACIONES ESTRATÉGICAS")
print("\n  1. Enfoque en tiendas de baja satisfacción:")
print("     - Implementar programas de capacitación al personal")
print("     - Mejorar la disponibilidad de productos")
print("     - Realizar auditorías de servicio al cliente")

print("\n  2. Optimización de inventarios:")
print(f"     - {len(inventario_critico)} productos en nivel crítico requieren revisión")
print("     - Implementar sistema de reorden automático")
print("     - Analizar demanda por producto y tienda")

print("\n  3. Proyecciones futuras:")
print(f"     - Ingresos esperados (12 meses): ${np.mean(ingresos_totales_simulados):,.2f}")
print(f"     - Probabilidad de crecimiento: {prob_crecimiento:.1f}%")
print(f"     - Intervalo de confianza 95%: ${ic_95_inferior:,.2f} - ${ic_95_superior:,.2f}")

print("\n" + "=" * 80)
print("FIN DEL ANÁLISIS")
print("=" * 80)